In [21]:
import json, pandas as pd

with open("big_dataset.json") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(df["intent"].value_counts())
print("Total:", len(df))

intent
greeting             105
job_search           103
out_of_scope         103
internship            99
experience_filter     82
help                  80
company_culture       76
apply                 69
resume                68
Name: count, dtype: int64
Total: 785


In [28]:
import nltk, re, string, numpy as np
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
KEEP = {'not','no','how','what','where','when','who','me','my','i'}
stop_words = stop_words - KEEP

def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    tokens = [lemmatizer.lemmatize(t) for t in text.split()
              if t not in stop_words]
    return ' '.join(tokens)

df['clean'] = df['context'].apply(preprocess)

X, y = df['clean'], df['intent']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, max_features=8000)),
        ("clf",   LogisticRegression(max_iter=1000, C=5, class_weight='balanced'))
    ]),
    "SVM": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, max_features=8000)),
        ("clf",   SVC(kernel='linear', C=1, probability=True, class_weight='balanced'))
    ]),
}

print("5-Fold Cross Validation:\n")
best_score, best_model = 0, None

for name, m in models.items():
    scores = cross_val_score(m, X, y, cv=5, scoring='accuracy')
    mean = round(scores.mean()*100, 2)
    std  = round(scores.std()*100, 2)
    print(f"  {name}: {mean}% ± {std}%")
    if mean > best_score:
        best_score, best_model, best_name = mean, m, name

print(f"\n Best: {best_name} ({best_score}%)")

5-Fold Cross Validation:

  Logistic Regression: 83.06% ± 1.54%
  SVM: 82.17% ± 1.14%

 Best: Logistic Regression (83.06%)


In [29]:
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
print(f"Test Accuracy: {round(accuracy_score(y_test, y_pred)*100,2)}%\n")
print(classification_report(y_test, y_pred))

Test Accuracy: 85.35%

                   precision    recall  f1-score   support

            apply       1.00      0.71      0.83        14
  company_culture       0.70      0.93      0.80        15
experience_filter       1.00      0.88      0.93        16
         greeting       1.00      0.76      0.86        21
             help       0.65      0.69      0.67        16
       internship       1.00      1.00      1.00        20
       job_search       0.90      0.90      0.90        21
     out_of_scope       0.74      0.85      0.79        20
           resume       0.81      0.93      0.87        14

         accuracy                           0.85       157
        macro avg       0.87      0.85      0.85       157
     weighted avg       0.87      0.85      0.86       157



In [30]:
from textblob import TextBlob

HINGLISH = {"kaise","karu","chahiye","mujhe","hai","kya","nahi","hain",
            "dikhao","batao","karo","milegi","madad","namaste","namaskar",
            "accha","theek","naukri","ke","liye","aur","wala"}

def is_hinglish(text):
    return any(w in HINGLISH for w in text.lower().split())

def correct(text):
    return text if is_hinglish(text) else str(TextBlob(text).correct())

def chat(text):
    original = text
    text = correct(text)
    if text != original:
        print(f"(corrected: '{original}' → '{text}')")
    clean = preprocess(text)
    intent = best_model.predict([clean])[0]
    conf   = round(best_model.predict_proba([clean]).max()*100, 1)
    if conf < 40: intent = "out_of_scope"

    responses = {
        "greeting":          "Hi! How can I help with your job search today?",
        "job_search":        "Here are job openings that match your search!",
        "internship":        "Here are our current internship listings!",
        "experience_filter": "Here are jobs matching your experience level!",
        "apply":             "Visit careers.mastercard.com and click 'Apply Now' on any role.",
        "resume":            "Upload your resume during the application process on our careers site.",
        "help":              "I can help with jobs, internships, applications, resume, and company info!",
        "company_culture":   "Mastercard has an inclusive culture, hybrid work, and great growth opportunities!",
        "out_of_scope":      "I only handle Mastercard career queries. Ask me about jobs or internships!"
    }
    print(f"You    : {original}")
    print(f"Intent : {intent} ({conf}%)")
    print(f"Bot    : {responses[intent]}")
    print("-"*50)

# All the cases original Mastercard bot failed
chat("hi")
chat("help me")
chat("what can you do")
chat("aplly job")
chat("job kaise apply karu")
chat("mujhe internship chahiye")
chat("rezume upload karna hai")
chat("work life balance kaisa hai")
chat("I am a fresher")
chat("i have 5 years experience show me jobs")
chat("order pizza")
chat("asdfgh")

You    : hi
Intent : greeting (97.9%)
Bot    : Hi! How can I help with your job search today?
--------------------------------------------------
You    : help me
Intent : help (83.4%)
Bot    : I can help with jobs, internships, applications, resume, and company info!
--------------------------------------------------
You    : what can you do
Intent : help (75.1%)
Bot    : I can help with jobs, internships, applications, resume, and company info!
--------------------------------------------------
(corrected: 'aplly job' → 'apply job')
You    : aplly job
Intent : apply (56.5%)
Bot    : Visit careers.mastercard.com and click 'Apply Now' on any role.
--------------------------------------------------
You    : job kaise apply karu
Intent : apply (63.2%)
Bot    : Visit careers.mastercard.com and click 'Apply Now' on any role.
--------------------------------------------------
You    : mujhe internship chahiye
Intent : internship (79.9%)
Bot    : Here are our current internship listings!
----

In [31]:
import pickle
with open("mastercard_bot_v3.pkl", "wb") as f:
    pickle.dump(best_model, f)
print("Saved")

Saved
